# Credit Card Fraud Detection using Machine Learning

## Oasis Infobyte Data Analytics Internship

### Level 2 – Task 3: Fraud Detection

**Author:** Rachaita Sarkar

**Tools Used**
- Python
- Pandas
- NumPy
- Scikit-learn
- SMOTE
- Matplotlib
- Seaborn

---

### Objective

Develop a machine learning pipeline to detect fraudulent credit card transactions from an imbalanced dataset.

The project aims to:

- Analyze the class imbalance in financial transactions.
- Perform Exploratory Data Analysis (EDA).
- Handle class imbalance using SMOTE.
- Build and compare multiple machine learning models.
- Evaluate models using Precision, Recall, F1-Score, and ROC-AUC.
- Identify the most important features contributing to fraud detection.
- Provide business recommendations for deploying a scalable fraud detection system.

---

# 1. Import Required Libraries

This section imports all the Python libraries required for data preprocessing, visualization, machine learning, model evaluation, and handling class imbalance.

The libraries include:

- Pandas and NumPy for data manipulation
- Matplotlib and Seaborn for visualization
- Scikit-learn for machine learning
- SMOTE from imbalanced-learn to balance the dataset

In [ ]:
# ============================================
# Import Required Libraries
# ============================================

import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE

plt.style.use("ggplot")

# Create images folder
os.makedirs("images", exist_ok=True)

print("All libraries imported successfully!")

# 2. Load Dataset

The Credit Card Fraud Detection dataset is loaded into a Pandas DataFrame for analysis.

The dataset contains anonymized financial transaction records where:

- **Class = 0** → Genuine Transaction
- **Class = 1** → Fraudulent Transaction

The dataset consists of transaction features, transaction amount, transaction time, and the target class.

In [ ]:
# ============================================
# Load Dataset
# ============================================

df = pd.read_csv("creditcard.csv")

print("Dataset loaded successfully!")

print("\nDataset Shape:")

print(df.shape)

display(df.head())

# 3. Initial Data Inspection

Before building any machine learning model, it is important to understand the structure and quality of the dataset.

In this section, we examine:

- Dataset dimensions
- Data types
- Missing values
- Summary statistics

This helps identify any preprocessing requirements before model development.

In [ ]:
# ============================================
# Initial Data Inspection
# ============================================

print("First Five Rows")
display(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nDataset Information")
df.info()

print("\nMissing Values")
print(df.isnull().sum())

print("\nDescriptive Statistics")
display(df.describe())

### Observation

From the initial inspection, the dataset contains numerical transaction features along with transaction time, amount, and the target variable (**Class**).

No missing values are present in the dataset, indicating that the data is already clean and suitable for machine learning after appropriate preprocessing.

# 4. Class Imbalance Analysis

Fraud detection datasets are naturally imbalanced because fraudulent transactions occur much less frequently than legitimate transactions.

This section examines:

- Number of genuine transactions
- Number of fraudulent transactions
- Percentage of fraud cases

Understanding the imbalance helps determine the appropriate evaluation metrics and motivates the use of SMOTE for balancing the training data.

In [ ]:
# ============================================
# Class Distribution
# ============================================

fraud_count = df["Class"].value_counts()

print("Transaction Counts")
print(fraud_count)

fraud_percentage = (fraud_count[1] / len(df)) * 100
non_fraud_percentage = (fraud_count[0] / len(df)) * 100

print(f"\nFraud Transactions     : {fraud_percentage:.4f}%")
print(f"Non-Fraud Transactions : {non_fraud_percentage:.4f}%")

In [ ]:
# ============================================
# Class Distribution Plot
# ============================================

plt.figure(figsize=(6,5))

sns.countplot(
    x="Class",
    data=df,
    palette="Set2"
)

plt.title("Fraud vs Non-Fraud Transactions")

plt.xlabel("Class (0 = Genuine, 1 = Fraud)")

plt.ylabel("Number of Transactions")

plt.savefig(
    "images/class_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Observation

The dataset is extremely imbalanced, with genuine transactions accounting for more than 99% of the observations and fraudulent transactions representing less than 1%.

Due to this imbalance, accuracy alone is not an appropriate evaluation metric. Precision, Recall, F1-Score, and ROC-AUC will be used later to assess the models more effectively.

# 5. Exploratory Data Analysis (EDA)

Exploratory Data Analysis (EDA) helps understand the underlying characteristics of the dataset before building machine learning models.

The following visualizations are used to analyze:

- Distribution of transaction amounts
- Comparison of transaction amounts between fraud and genuine transactions
- Distribution of transaction time
- Relationships among numerical features using a correlation heatmap

In [ ]:
# ============================================
# Transaction Amount Distribution
# ============================================

plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x="Amount",
    bins=50,
    kde=True,
    color="steelblue"
)

plt.title("Distribution of Transaction Amount")

plt.xlabel("Transaction Amount")

plt.ylabel("Frequency")

plt.savefig(
    "images/amount_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Observation

The transaction amount distribution is highly right-skewed. Most transactions involve relatively small amounts, while only a few transactions have very high values.

This indicates that unusually large transactions are rare and may require additional monitoring during fraud detection.

### Time-of-Day Analysis

The `Time` variable records the number of seconds elapsed since the first transaction in the dataset.

For better interpretation, it is converted into hours to analyze how transaction activity varies throughout the day.

In [ ]:
# ============================================
# Time-of-Day Analysis
# ============================================

df["Hour"] = (df["Time"] // 3600) % 24

plt.figure(figsize=(10,5))

sns.countplot(
    x="Hour",
    data=df,
    color="steelblue"
)

plt.title("Transactions by Hour of the Day")

plt.xlabel("Hour")

plt.ylabel("Number of Transactions")

plt.savefig(
    "images/time_of_day_analysis.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Observation

Transaction activity varies across different hours of the day. Such temporal patterns can provide additional information for fraud detection systems, as fraudulent transactions may occur more frequently during specific time periods.

### Transaction Amount by Transaction Class

This box plot compares the transaction amounts of genuine and fraudulent transactions to identify differences in their spending behaviour.

In [ ]:
# ============================================
# Transaction Amount by Class
# ============================================

plt.figure(figsize=(8,6))

sns.boxplot(
    data=df,
    x="Class",
    y="Amount"
)

plt.title("Transaction Amount: Fraud vs Genuine")

plt.xlabel("Class (0 = Genuine, 1 = Fraud)")

plt.ylabel("Transaction Amount")

plt.savefig(
    "images/fraud_amount_boxplot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Observation

Fraudulent transactions generally involve lower transaction amounts than genuine transactions, although there is some overlap between the two classes.

The presence of outliers in both groups indicates that high-value transactions can occur in both legitimate and fraudulent activities.

### Transaction Time Distribution

This histogram shows how transactions are distributed over time and helps identify periods with higher transaction activity.

In [ ]:
# ============================================
# Transaction Time Distribution
# ============================================

plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x="Time",
    bins=50,
    color="orange"
)

plt.title("Transaction Time Distribution")

plt.xlabel("Time")

plt.ylabel("Frequency")

plt.savefig(
    "images/time_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Observation

Transactions occur continuously throughout the recorded time period. Some intervals exhibit higher transaction frequencies, indicating periods of increased customer activity.

The time variable may contribute useful information when distinguishing fraudulent transactions from genuine ones.

### Correlation Analysis

A correlation heatmap is used to examine relationships among numerical variables and identify features that may influence fraud detection.

In [ ]:
# ============================================
# Correlation Heatmap
# ============================================

plt.figure(figsize=(16,12))

sns.heatmap(
    df.corr(),
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Matrix")

plt.savefig(
    "images/correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# 6. Feature Selection

Before training machine learning models, the dataset is divided into:

- **Features (X):** All predictor variables used to identify fraud.
- **Target Variable (y):** The `Class` column, where:
  - **0** = Genuine Transaction
  - **1** = Fraudulent Transaction

Separating the features and target variable prepares the data for model training.

In [ ]:
# ============================================
# Feature Selection
# ============================================

X = df.drop("Class", axis=1)
y = df["Class"]

print("Feature Matrix Shape :", X.shape)
print("Target Variable Shape:", y.shape)

print("\nTarget Variable Distribution:")
print(y.value_counts())

### Observation

The dataset has been successfully divided into predictor variables (`X`) and the target variable (`y`). The target variable remains highly imbalanced, reinforcing the need for specialized techniques such as SMOTE.

# 7. Train-Test Split

The dataset is divided into training and testing sets using an **80:20 ratio**.

A **stratified split** is applied to ensure that both training and testing datasets maintain the original proportion of fraudulent and genuine transactions.

This is particularly important for highly imbalanced datasets.

In [ ]:
# ============================================
# Train-Test Split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Samples :", X_train.shape[0])
print("Testing Samples  :", X_test.shape[0])

print("\nTraining Class Distribution:")
print(y_train.value_counts())

print("\nTesting Class Distribution:")
print(y_test.value_counts())

### Observation

The training and testing datasets preserve the original fraud-to-genuine transaction ratio due to stratified sampling. This helps ensure reliable model evaluation and prevents bias caused by uneven class distributions.

# 8. Feature Scaling

Machine learning algorithms often perform better when numerical features are on a similar scale.

The **StandardScaler** standardizes each feature by transforming it to have:

- Mean = 0
- Standard Deviation = 1

Scaling is especially beneficial for algorithms such as Logistic Regression.

In [ ]:
# ============================================
# Feature Scaling
# ============================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed successfully.")

### Observation

All numerical features have been standardized, ensuring that no single feature dominates the learning process because of its scale.

# 9. Handling Class Imbalance using SMOTE

The Credit Card Fraud Detection dataset contains significantly fewer fraudulent transactions than genuine transactions.

To overcome this imbalance, **SMOTE (Synthetic Minority Over-sampling Technique)** is applied **only to the training dataset**.

SMOTE creates synthetic examples of the minority class, allowing the machine learning models to learn fraud patterns more effectively while preventing information leakage into the testing data.

In [ ]:
# ============================================
# Apply SMOTE
# ============================================

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

print("Original Training Shape :", X_train.shape)
print("Balanced Training Shape :", X_train_smote.shape)

print("\nClass Distribution After SMOTE:")
print(pd.Series(y_train_smote).value_counts())

### Observation

After applying SMOTE, the training dataset contains an equal number of genuine and fraudulent transactions.

Balancing the classes enables the machine learning algorithms to learn fraud-related patterns more effectively and reduces bias toward the majority class.

# 10. Logistic Regression Model

Logistic Regression is a supervised machine learning algorithm widely used for binary classification problems.

In this project, it serves as the baseline model for classifying transactions as either:

- **0 → Genuine Transaction**
- **1 → Fraudulent Transaction**

The model is trained using the balanced dataset generated by SMOTE.

In [ ]:
# ============================================
# Logistic Regression Model
# ============================================

lr = LogisticRegression(
    random_state=42,
    max_iter=1000
)

lr.fit(X_train_smote, y_train_smote)

lr_predictions = lr.predict(X_test_scaled)

print("Logistic Regression model trained successfully.")

### Observation

The Logistic Regression model has been successfully trained using the balanced training dataset. The next step is to evaluate its performance using multiple classification metrics.

In [ ]:
# ============================================
# Logistic Regression Evaluation
# ============================================

lr_accuracy = accuracy_score(y_test, lr_predictions)
lr_precision = precision_score(y_test, lr_predictions)
lr_recall = recall_score(y_test, lr_predictions)
lr_f1 = f1_score(y_test, lr_predictions)

print("Logistic Regression Performance")
print("-" * 40)

print(f"Accuracy : {lr_accuracy:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall   : {lr_recall:.4f}")
print(f"F1 Score : {lr_f1:.4f}")

print("\nClassification Report\n")

print(classification_report(y_test, lr_predictions))

### Observation

Although the model achieves a high accuracy score, accuracy alone is not sufficient for evaluating fraud detection systems due to the highly imbalanced nature of the dataset.

Recall is especially important because failing to identify a fraudulent transaction may result in significant financial losses.

In [ ]:
# ============================================
# Confusion Matrix
# ============================================

cm = confusion_matrix(y_test, lr_predictions)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Genuine", "Fraud"]
)

disp.plot(cmap="Blues")

plt.title("Logistic Regression Confusion Matrix")

plt.savefig(
    "images/logistic_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Observation

The confusion matrix summarizes the model's predictions by showing correctly classified and misclassified transactions.

The primary objective is to minimize **False Negatives**, as these represent fraudulent transactions incorrectly classified as genuine.

In [ ]:
# ============================================
# ROC Curve and AUC Score
# ============================================

lr_probabilities = lr.predict_proba(X_test_scaled)[:, 1]

lr_auc = roc_auc_score(y_test, lr_probabilities)

fpr, tpr, thresholds = roc_curve(y_test, lr_probabilities)

plt.figure(figsize=(7,6))

plt.plot(
    fpr,
    tpr,
    label=f"Logistic Regression (AUC = {lr_auc:.4f})"
)

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve - Logistic Regression")

plt.legend()

plt.savefig(
    "images/logistic_roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"ROC-AUC Score: {lr_auc:.4f}")

### Observation

The ROC curve illustrates the trade-off between the True Positive Rate and the False Positive Rate across different classification thresholds.

A higher ROC-AUC score indicates stronger discrimination between fraudulent and genuine transactions, reflecting better overall model performance.

# 11. Random Forest Model

Random Forest is an ensemble learning algorithm that combines multiple decision trees to improve prediction accuracy and reduce overfitting.

Compared with Logistic Regression, Random Forest is capable of capturing complex, non-linear relationships within the data, making it a strong candidate for fraud detection.

In [ ]:
# ============================================
# Random Forest Model
# ============================================

rf = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_smote, y_train_smote)

rf_predictions = rf.predict(X_test_scaled)

print("Random Forest model trained successfully.")

### Observation

The Random Forest model has been trained successfully using the balanced dataset generated through SMOTE. The next step is to evaluate its performance using multiple classification metrics.

In [ ]:
# ============================================
# Random Forest Evaluation
# ============================================

rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_precision = precision_score(y_test, rf_predictions)
rf_recall = recall_score(y_test, rf_predictions)
rf_f1 = f1_score(y_test, rf_predictions)

print("Random Forest Performance")
print("-" * 40)

print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1 Score : {rf_f1:.4f}")

print("\nClassification Report\n")

print(classification_report(y_test, rf_predictions))

### Observation

The Random Forest model generally provides stronger predictive performance than Logistic Regression by capturing complex relationships among the transaction features.

Its Precision, Recall, and F1-Score provide a more reliable assessment than overall accuracy for fraud detection.

In [ ]:
# ============================================
# Random Forest Confusion Matrix
# ============================================

cm_rf = confusion_matrix(y_test, rf_predictions)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_rf,
    display_labels=["Genuine","Fraud"]
)

disp.plot(cmap="Greens")

plt.title("Random Forest Confusion Matrix")

plt.savefig(
    "images/random_forest_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Observation

The confusion matrix illustrates the classification performance of the Random Forest model.

A lower number of False Negatives indicates improved fraud detection capability, while a lower number of False Positives reduces unnecessary transaction investigations.

In [ ]:
# ============================================
# ROC Curve - Random Forest
# ============================================

rf_probabilities = rf.predict_proba(X_test_scaled)[:,1]

rf_auc = roc_auc_score(
    y_test,
    rf_probabilities
)

fpr_rf, tpr_rf, thresholds_rf = roc_curve(
    y_test,
    rf_probabilities
)

plt.figure(figsize=(7,6))

plt.plot(
    fpr_rf,
    tpr_rf,
    label=f"Random Forest (AUC = {rf_auc:.4f})",
    linewidth=2
)

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve - Random Forest")

plt.legend()

plt.savefig(
    "images/random_forest_roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"ROC-AUC Score : {rf_auc:.4f}")

### Observation

The ROC curve demonstrates the Random Forest model's ability to distinguish between fraudulent and genuine transactions across multiple classification thresholds.

A ROC-AUC value closer to 1 indicates excellent classification performance.

In [ ]:
# ============================================
# Feature Importance
# ============================================

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

display(importance.head(10))

plt.figure(figsize=(10,6))

sns.barplot(
    data=importance.head(10),
    x="Importance",
    y="Feature",
    palette="viridis"
)

plt.title("Top 10 Important Features")

plt.savefig(
    "images/feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Observation

The feature importance analysis identifies the variables that contribute most to predicting fraudulent transactions.

These features play a significant role in helping the Random Forest model distinguish fraudulent behavior from legitimate transactions.

# 12. Model Comparison

After training both Logistic Regression and Random Forest models, their performance is compared using multiple evaluation metrics.

Since fraud detection is a highly imbalanced classification problem, metrics such as Precision, Recall, F1-Score, and ROC-AUC provide a more meaningful evaluation than Accuracy alone.

In [ ]:
# ============================================
# Model Comparison
# ============================================

comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [lr_accuracy, rf_accuracy],
    "Precision": [lr_precision, rf_precision],
    "Recall": [lr_recall, rf_recall],
    "F1 Score": [lr_f1, rf_f1],
    "ROC-AUC": [lr_auc, rf_auc]
})

display(comparison)

comparison.to_csv("Model_Comparison.csv", index=False)

### Observation

The comparison table summarizes the performance of both machine learning models.

The model with higher Recall and ROC-AUC is generally more suitable for fraud detection because correctly identifying fraudulent transactions is more important than achieving the highest overall accuracy.

# 13. Accuracy Discussion

Although both models achieve high accuracy, accuracy is not an appropriate standalone metric for fraud detection.

Since fraudulent transactions represent only a very small percentage of the dataset, a model can achieve very high accuracy simply by predicting every transaction as genuine.

Therefore, additional evaluation metrics such as Precision, Recall, F1-Score, and ROC-AUC are necessary to measure the true effectiveness of the model.

# 14. Recall vs Precision Discussion

Recall measures the proportion of fraudulent transactions that are correctly identified by the model.

Precision measures the proportion of transactions predicted as fraudulent that are actually fraudulent.

In fraud detection, Recall is generally considered more important because failing to detect a fraudulent transaction can result in significant financial losses.

However, excessively high Recall may increase False Positives, leading to unnecessary investigations.

An effective fraud detection system should maintain an appropriate balance between Recall and Precision.

# 15. Feature Importance Discussion

The Random Forest model provides feature importance scores that indicate which variables contribute the most to fraud prediction.

These important features can help financial institutions better understand suspicious transaction behavior and improve future fraud detection systems.

# 16. Scalability Discussion

A production-level fraud detection system may need to process millions of transactions every hour.

To support such workloads, the model could be deployed using distributed computing frameworks such as Apache Spark or cloud platforms including AWS, Microsoft Azure, or Google Cloud Platform.

Real-time prediction APIs and streaming technologies such as Apache Kafka could further improve scalability and enable continuous fraud monitoring.

# 17. Conclusion

This project successfully developed a machine learning pipeline for detecting fraudulent credit card transactions.

Key accomplishments include:

- Performing Exploratory Data Analysis (EDA)
- Analyzing severe class imbalance
- Applying SMOTE to balance the training data
- Training Logistic Regression and Random Forest models
- Evaluating models using Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and ROC-AUC
- Identifying the most influential features for fraud prediction

Among the evaluated models, Random Forest demonstrated superior performance and is recommended for fraud detection applications.

# 18. Business Recommendations

Based on the findings of this analysis, the following recommendations are proposed:

1. Deploy the Random Forest model for fraud detection due to its superior predictive performance.

2. Continuously retrain the model using newly available transaction data to adapt to evolving fraud patterns.

3. Use Recall as the primary evaluation metric to minimize missed fraudulent transactions.

4. Integrate the model into a real-time transaction monitoring system to detect suspicious activities immediately.

5. Combine machine learning predictions with rule-based fraud detection systems to further improve reliability and reduce financial risk.

In [ ]:
# ============================================
# Save Final Outputs
# ============================================

comparison.to_csv("Model_Comparison.csv", index=False)

print("Model comparison saved as Model_Comparison.csv")

print("\nProject completed successfully!")

print("Generated Outputs:")
print("- Model_Comparison.csv")
print("- Images saved in the images folder")

## 1. Import Required Libraries

This section imports all the Python libraries required for data analysis, visualization, preprocessing, model building, and evaluation.

In [ ]:
# ============================================
# Import Required Libraries
# ============================================

import os
import warnings

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

plt.style.use("ggplot")

# Create images folder if it doesn't exist
os.makedirs("images", exist_ok=True)

## 2. Load Dataset

The Credit Card Fraud Detection dataset is loaded into a pandas DataFrame for further analysis.

In [ ]:
# ============================================
# Load Dataset
# ============================================

df = pd.read_csv("creditcard.csv")

print("Dataset Loaded Successfully!")

## 3. Initial Data Inspection

The dataset is inspected to understand its dimensions, data types, missing values, and descriptive statistics before preprocessing.

In [ ]:
# ============================================
# Initial Data Inspection
# ============================================

print("First Five Rows")
display(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nDataset Information")
df.info()

print("\nMissing Values")
print(df.isnull().sum())

print("\nDescriptive Statistics")
display(df.describe())

## 4. Class Imbalance Analysis

Fraud detection datasets are highly imbalanced. This section examines the percentage of fraudulent and genuine transactions.

In [ ]:
# ============================================
# Class Distribution
# ============================================

fraud_count = df["Class"].value_counts()

print(fraud_count)

fraud_percentage = (fraud_count[1] / len(df)) * 100
non_fraud_percentage = (fraud_count[0] / len(df)) * 100

print(f"\nFraud Transactions: {fraud_percentage:.4f}%")
print(f"Non-Fraud Transactions: {non_fraud_percentage:.4f}%")

In [ ]:
# ============================================
# Class Distribution Plot
# ============================================

plt.figure(figsize=(6,5))

sns.countplot(
    x="Class",
    data=df,
    palette="Set2"
)

plt.title("Fraud vs Non-Fraud Transactions")

plt.xlabel("Class (0 = Genuine, 1 = Fraud)")

plt.ylabel("Number of Transactions")

plt.savefig("images/class_distribution.png",
            dpi=300,
            bbox_inches="tight")

plt.show()

### Observation

The dataset is extremely imbalanced, with fraudulent transactions accounting for less than 1% of all records. This imbalance makes accuracy alone an unreliable evaluation metric.

## 5. Exploratory Data Analysis (EDA)

Exploratory Data Analysis helps identify trends, distributions, and relationships within the dataset before machine learning.

In [ ]:
# ============================================
# Transaction Amount Distribution
# ============================================

plt.figure(figsize=(8,5))

sns.histplot(df['Amount'], bins=50, kde=True)

plt.title("Distribution of Transaction Amount")

plt.xlabel("Transaction Amount")

plt.ylabel("Frequency")

plt.savefig("images/amount_distribution.png",
            dpi=300,
            bbox_inches="tight")

plt.show()

### Observation

Most transaction amounts are relatively small, while only a few transactions involve very high values, resulting in a right-skewed distribution.

### Transaction Amount by Transaction Class

This box plot compares the transaction amounts of genuine and fraudulent transactions to identify differences in their spending patterns.

In [ ]:
# ============================================
# Transaction Amount by Class
# ============================================

plt.figure(figsize=(8,6))

sns.boxplot(
    x='Class',
    y='Amount',
    data=df
)

plt.title("Transaction Amount: Fraud vs Non-Fraud")

plt.xlabel("Class (0 = Genuine, 1 = Fraud)")

plt.ylabel("Amount")

plt.savefig("images/fraud_amount_boxplot.png",
            dpi=300,
            bbox_inches="tight")

plt.show()

### Observation

Fraudulent transactions generally involve lower transaction amounts than genuine transactions, although some overlap exists. The presence of outliers indicates that both fraud and legitimate transactions can occasionally involve high-value purchases.

### Transaction Time Distribution

This histogram shows how transactions are distributed over time and helps identify periods with higher transaction activity.

In [ ]:
# ============================================
# Transaction Time Distribution
# ============================================

plt.figure(figsize=(8,5))

sns.histplot(df['Time'], bins=50)

plt.title("Transaction Time Distribution")

plt.xlabel("Time")

plt.ylabel("Frequency")

plt.savefig("images/time_distribution.png",
            dpi=300,
            bbox_inches="tight")

plt.show()

### Observation

Transactions occur throughout the recorded time period, indicating continuous customer activity. Certain periods exhibit higher transaction volumes, suggesting variations in transaction frequency over time.

### Correlation Analysis

A correlation heatmap is used to examine the relationships between numerical features and identify variables that may be useful for fraud detection.

In [ ]:
# ============================================
# Correlation Heatmap
# ============================================

plt.figure(figsize=(15,10))

sns.heatmap(
    df.corr(),
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Matrix")

plt.savefig("images/correlation_heatmap.png",
            dpi=300,
            bbox_inches="tight")

plt.show()

### Observation

Most anonymized features show weak correlations with one another. However, a few variables exhibit stronger relationships with the target variable (Class), making them valuable predictors for identifying fraudulent transactions.

## 6. Feature Selection

The dataset is divided into predictor variables (X) and the target variable (y). The target variable, **Class**, indicates whether a transaction is genuine (0) or fraudulent (1).

In [ ]:
# ============================================
# Feature Selection
# ============================================

X = df.drop("Class", axis=1)

y = df["Class"]

print("Feature Matrix Shape:", X.shape)
print("Target Variable Shape:", y.shape)

## 7. Train-Test Split

The dataset is divided into training (80%) and testing (20%) sets using stratified sampling to maintain the original class distribution in both subsets.

In [ ]:
# ============================================
# Train-Test Split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

## 8. Feature Scaling

StandardScaler is applied to standardize the numerical features so that each feature has a mean of 0 and a standard deviation of 1. This improves the performance of many machine learning algorithms.

In [ ]:
# ============================================
# Feature Scaling
# ============================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

## 9. Handling Class Imbalance using SMOTE

The dataset contains significantly fewer fraudulent transactions than genuine ones. SMOTE (Synthetic Minority Oversampling Technique) is applied to the training data to generate synthetic fraud samples and balance the class distribution.

In [ ]:
# ============================================
# Apply SMOTE
# ============================================

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

print("Original Training Shape:", X_train.shape)
print("Balanced Training Shape:", X_train_smote.shape)

print("\nClass Distribution After SMOTE")

print(pd.Series(y_train_smote).value_counts())

### Observation

After applying SMOTE, both classes have an equal number of samples in the training dataset. This helps the machine learning models learn fraud patterns more effectively and reduces bias toward the majority class.

## 10. Logistic Regression Model

Logistic Regression is used as a baseline classification algorithm to distinguish between fraudulent and genuine transactions.

In [ ]:
# ============================================
# Logistic Regression
# ============================================

lr = LogisticRegression(
    random_state=42,
    max_iter=1000
)

lr.fit(X_train_smote, y_train_smote)

lr_predictions = lr.predict(X_test_scaled)

In [ ]:
# ============================================
# Logistic Regression Evaluation
# ============================================

print("Accuracy :", accuracy_score(y_test, lr_predictions))

print("Precision :", precision_score(y_test, lr_predictions))

print("Recall :", recall_score(y_test, lr_predictions))

print("F1 Score :", f1_score(y_test, lr_predictions))

print("\nClassification Report\n")

print(classification_report(y_test, lr_predictions))

### Observation

The Logistic Regression model demonstrates strong fraud detection capability. In fraud detection, Recall is particularly important because identifying fraudulent transactions is often more critical than maximizing overall accuracy.

In [ ]:
# ============================================
# Random Forest Model
# ============================================

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train_smote, y_train_smote)

rf_predictions = rf.predict(X_test_scaled)